### GridWorld environment

`step(cell, action)` takes the cell as an argument and returns the next one. the class holds no
position, so we can ask about any cell at any time, in any order.

that's exactly what DP needs. a simulator would only answer "what's next from where I am now".

walking into a wall or off the edge keeps you in place. reward comes from the cell you land on,
so a terminal pays out when you enter it.

In [1]:
UP, RIGHT, DOWN, LEFT = 'up', 'right', 'down', 'left'
ACTIONS = [UP, RIGHT, DOWN, LEFT]
DIRECTIONS = {UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0), LEFT: (0, -1)}
ARROWS = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}

In [2]:
class GridWorld:

    def __init__(
        self,
        rows: int,
        cols: int,
        step_reward: float,
        terminals: dict[tuple[int, int], float],
        walls: set[tuple[int, int]],
    ):
        self.rows = rows
        self.cols = cols
        self.step_reward = step_reward
        self.terminals = terminals
        self.walls = walls

    def step(self, cell: tuple[int, int], action: str):
        dr, dc = DIRECTIONS[action]
        next_cell = (cell[0] + dr, cell[1] + dc)
        outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
        on_wall = next_cell in self.walls
        if outside or on_wall:
            next_cell = cell
        if next_cell in self.terminals:
            reward = self.terminals[next_cell]
        else:
            reward = self.step_reward
        return next_cell, reward

    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return f"{self.terminals[cell]:+}"
        elif cell in self.walls:
            return "#"
        else:
            return "·"

    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print("".join([f"{v:>3} " for v in range(self.cols)]))
                if c == 0:
                    print(f"{r:>3} ", end="")
                print(f"{self.cell_repr(r, c):>3} ", end="")
            print()

    def __repr__(self):
        return f"""
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
"""

In [3]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)
env


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

step_reward is 0, so wandering costs nothing. the only pressure to reach the goal is gamma —
a +1 that is 3 steps away is worth 0.9^3.

In [4]:
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


In [5]:
cell = (0, 0)
for _ in range(3):
    cell, reward = env.step(cell, RIGHT)
    print((cell, reward))

((0, 1), 0)
((0, 2), 0)
((0, 3), 1)


### value_iteration

one sweep = update every cell once, using the previous sweep's V. `V_old` is that snapshot,
so the order we visit cells in doesn't change the result.

walls and terminals are skipped and stay 0. a terminal's reward is already paid on the way in,
holding a value there too would count it twice.

delta is the largest change in the sweep. when nothing moves, we are at the fixed point.

In [6]:
def show_V(env: GridWorld, V: list[list[float]]):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print("".join([f"{v:>4} " for v in range(env.cols)]))
            if c == 0:
                print(f"{r:>4} ", end="")

            value = round(V[r][c], 2)
            print(f"{value:>4} ", end="")
        print()

In [7]:
def value_iteration(env: GridWorld, gamma=0.9, theta=1e-6):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    print('V init')
    show_V(env, V)
    print('-' * 25) 
    delta = float("inf")
    i = 0
    while delta >= theta:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                max_value = float("-inf")
                for action in ACTIONS:
                    next_cell, reward = env.step(cell, action)
                    action_value = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                    max_value = max(max_value, action_value)
                V[r][c] = max_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        print(f"iter {i}: delta={delta:.6f}")
        show_V(env, V)
        print("-" * 25)
        i += 1
    print(f"value_iteration converged in {i} iterations")
    return V

In [8]:
#| anchor: value-iteration-trace
V = value_iteration(env)

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.

### value spreads one cell per sweep

iter 0 fills only (0,2). iter 1 adds its neighbours, iter 2 the next ring. information moves
one cell per sweep, so the sweep count is set by the distance from the goal to the furthest cell.

delta drops by exactly gamma each sweep — 1.0, 0.9, 0.81, 0.729, 0.656 — and then goes straight
to 0. transitions are deterministic, so once value reaches the furthest cell there is nothing
left to refine. no slow tail.

theta never gets used. delta goes from 0.656 straight to 0, so the loop stops on an exact
fixed point, not on the threshold.

In [9]:
show_V(env, V)

  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2 0.66 0.73 0.81 0.73 


every value is gamma^(steps to goal - 1). (0,2)=1.0, (0,1)=0.9, (0,0)=0.81, (2,0)=0.66.
V is a distance map written in gamma.

(2,3) sits right next to the -1 and is still 0.73, because the policy there goes left.
V scores the plan, not the neighbourhood.

### read_policy

same backup as value_iteration, but keeping the argmax instead of the max.

#| anchor: ties-go-to-actions-order

the comparison is a strict `>`, so on a tie the first action in `ACTIONS` wins — that's UP.
the answer then depends on the order of `ACTIONS`, not on V alone.

In [10]:
def read_policy(env: GridWorld, V: list[list[float]], gamma=0.9):
    policy = [[None for _ in range(env.cols)] for _ in range(env.rows)]
    for r in range(env.rows):
        for c in range(env.cols):
            cell = (r, c)
            if cell in env.walls or cell in env.terminals:
                continue

            max_value = float("-inf")
            for action in ACTIONS:
                next_cell, reward = env.step(cell, action)
                action_value = reward + gamma * V[next_cell[0]][next_cell[1]]
                if action_value > max_value:
                    policy[r][c] = action
                    max_value = action_value
    return policy

there is a tie in this grid. at (2,0), UP leads to (1,0) and RIGHT leads to (2,1), and both
are worth 0.729 — so both actions score 0 + 0.9 * 0.729 = 0.6561.

`>` keeps whichever came first, so the output below should say up at (2,0).

In [11]:
policy = read_policy(env, V)
policy

[['right', 'right', 'right', None],
 ['up', None, 'up', None],
 ['up', 'right', 'up', 'left']]

In [12]:
def render_policy(env: GridWorld, policy: list[list[str | None]]):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print(" r/c", end="")
                print("".join([f"{v:>3} " for v in range(env.cols)]))
            if c == 0:
                print(f"{r:>3} ", end="")

            cell = (r, c)
            if cell in env.walls:
                value = '#'
            elif cell in env.terminals:
                value = f"{env.terminals[cell]:+}"
            elif policy[r][c] is not None:
                value = ARROWS[policy[r][c]]
            else:
                raise ValueError(f"no action for cell {cell}")

            print(f"{value:>3} ", end="")
        print()

In [13]:
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 


### read policy during value_iteration 

we can also read policy during value_iteration, no need to do it with returned V

the max already looks at all 4 actions, so remembering which one won costs nothing. calling
read_policy afterwards repeats a full sweep of work that was already done, and thrown away.

In [14]:
def value_iteration(env: GridWorld, gamma=0.9, theta=1e-6):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    print('V init')
    show_V(env, V)
    print('-' * 25)  
    policy = [[None for _ in range(env.cols)] for _ in range(env.rows)]
    delta = float("inf")
    i = 0
    while delta >= theta:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                max_value = float("-inf")
                for action in ACTIONS:
                    next_cell, reward = env.step(cell, action)
                    action_value = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                    if action_value > max_value:
                        max_value = action_value
                        policy[r][c] = action
                V[r][c] = max_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        print(f"iter {i}: delta={delta:.6f}")
        show_V(env, V)
        print("-" * 25)
        i += 1
    print('policy:')
    render_policy(env, policy)
    print("-" * 25)
    print(f"value_iteration converged in {i} iterations")
    return policy, V

In [15]:
policy, V = value_iteration(env)

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.

### policy_evaluation behaves same as value_iteration under optimal policy

the only difference from value_iteration is the max. here the policy names the action, so it's
one backup per cell instead of four.

given the optimal policy it lands on the same V in the same 6 sweeps. it has to — the max was
picking exactly the action the policy already names.

In [16]:
def policy_evaluation(
    env: GridWorld,
    policy: list[list[str | None]],
    gamma=0.9,
    theta=1e-6,
    verbose=True
):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    if verbose:
        print('V init')
        show_V(env, V)
        print('-' * 25)  
    delta = float("inf")
    i = 0
    while delta >= theta:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                action = policy[r][c]
                next_cell, reward = env.step(cell, action)
                V[r][c] = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            print(f"iter {i}: delta={delta:.6f}")
            show_V(env, V)
            print("-" * 25)
        i += 1
    print(f"policy_evaluation converged in {i} iterations")
    return V

In [17]:
V = policy_evaluation(env, policy)

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.

### policy_iteration

start from any policy (all UP here), evaluate it, read a greedy policy off that V, repeat.

read_policy runs once per round, so ↑ [the ACTIONS tie-break](#ties-go-to-actions-order)
applies every round, not just at the end.

the stopping rule is different from value_iteration: it stops when no action changes, not when
V stops changing.

actions changed goes 2, 2, 2, 1, 0. the policy was already final a round before the loop ended —
the last round exists only to prove nothing changed.

policy_iteration says 5 iterations and ↑ [value_iteration](#value_iteration) says [6](#value-iteration-trace),
but those are not the same unit.
one policy_iteration round contains a whole evaluation — the counts printed below are
2, 5, 5, 6, 6 sweeps.

In [18]:
def policy_iteration(env: GridWorld, gamma=0.9, theta=1e-6, pe_verbose=False):
    policy = [[UP for _ in range(env.cols)] for _ in range(env.rows)]
    changed = None
    i = 0
    while changed != 0:
        print(f'iter {i}:')
        V = policy_evaluation(env, policy, gamma, theta, verbose=pe_verbose)
        new_policy = read_policy(env, V, gamma)
        changed = 0
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                if new_policy[r][c] != policy[r][c]:
                    changed += 1
        print(f'V:')
        show_V(env, V)
        print('-' * 25)
        print(f'actions changed = {changed}')
        render_policy(env, new_policy)
        print('-' * 5)
        policy = new_policy
        i += 1
    print(f'policy_iteration converged in {i} iterations')
    return policy, V

In [19]:
policy, V = policy_iteration(env)

iter 0:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-----
iter 1:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-----
iter 2:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-----
iter 3:
policy_evaluation converged in 6 iterations
V:
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.7

### don't update action until better is available

currently tied action at cell (2, 0), get's flipped from RIGHT to UP, we can make it so that action isn't updated until strictly better is available

with it, policy_iteration lands in 4 rounds instead of 5. the extra round was spent flipping a tie and flipping it back (last code cell op iter 3).

In [20]:
def read_policy(
    env: GridWorld,
    V: list[list[float]],
    gamma=0.9,
    theta=1e-6,
    incumbent_policy: list[list[str | None]] | None = None,
):
    policy = [[None for _ in range(env.cols)] for _ in range(env.rows)]
    for r in range(env.rows):
        for c in range(env.cols):
            cell = (r, c)
            if cell in env.walls or cell in env.terminals:
                continue
            if incumbent_policy:
                incumbent_action = incumbent_policy[r][c]
                if incumbent_action:
                    next_cell, reward = env.step(cell, incumbent_action)
                    incumbent_action_value = (
                        reward + gamma * V[next_cell[0]][next_cell[1]]
                    )

            max_value = float("-inf")
            for action in ACTIONS:
                next_cell, reward = env.step(cell, action)
                action_value = reward + gamma * V[next_cell[0]][next_cell[1]]
                if action_value > max_value:
                    if (
                        incumbent_policy
                        and incumbent_action
                        and action_value < incumbent_action_value + theta
                    ):
                        action = incumbent_action
                        action_value = incumbent_action_value
                    policy[r][c] = action
                    max_value = action_value
    return policy

In [21]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    pass_incumbent_policy=True,
    pe_verbose=False,
):
    policy = [[UP for _ in range(env.cols)] for _ in range(env.rows)]
    changed = None
    i = 0
    while changed != 0 and i < max_iters:
        print(f'iter {i}:')
        V = policy_evaluation(env, policy, gamma, theta, verbose=pe_verbose)
        new_policy = read_policy(
            env,
            V,
            gamma,
            theta,
            incumbent_policy=policy if pass_incumbent_policy else None,
        )
        changed = 0
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                if new_policy[r][c] != policy[r][c]:
                    changed += 1
        print(f'V:')
        show_V(env, V)
        print('-' * 25)
        print(f'actions changed = {changed}')
        render_policy(env, new_policy)
        print('-' * 5)
        policy = new_policy
        i += 1
    converged = changed == 0
    if converged:
        print(f'policy_iteration converged in {i} iterations')
    else:
        print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [22]:
policy, V, converged = policy_iteration(env, pass_incumbent_policy=True)

iter 0:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-----
iter 1:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-----
iter 2:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-----
iter 3:
policy_evaluation converged in 6 iterations
V:
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.7

### policy iteration keeps flipping up/right on cell (0, 2) without pass_incumbent_policy

at gamma=1.0 with step_reward=0 every path to the goal is worth the same, so ties are everywhere,
not just at one cell.

without pass_incumbent_policy the greedy read keeps swapping between two tied actions and the loop
never ends — 100 rounds and still going. with it, 4 rounds.

the fix is not about finding a better policy. both actions really are equally good. it is about
not calling a tie a change.

In [23]:
policy, V, converged = policy_iteration(
    env, gamma=1.0, max_iters=100, pass_incumbent_policy=False
)

iter 0:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-----
iter 1:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  1.0  0.0 
   2  0.0  0.0  1.0  1.0 
-------------------------
actions changed = 4
 r/c  0   1   2   3 
  0   ↑   →   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   → 
-----
iter 2:
policy_evaluation converged in 1 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
actions changed = 3
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   → 
-----
iter 3:
policy_evaluation converged in 4 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.

In [24]:
policy, V, converged = policy_iteration(env, gamma=1.0, pass_incumbent_policy=True)

iter 0:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-----
iter 1:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  1.0  0.0 
   2  0.0  0.0  1.0  1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-----
iter 2:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  1.0  1.0  0.0 
   1  0.0  0.0  1.0  0.0 
   2  0.0  1.0  1.0  1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-----
iter 3:
policy_evaluation converged in 6 iterations
V:
  r/c   0    1    2    3 
   0  1.0  1.0  1.0  0.0 
   1  1.

### step reward -1, without discount produces steps to goal behaviour

step_reward=-1 with gamma=1.0 makes V exactly -(steps to the exit). nothing is discounted, so every
step costs 1 and the value is a step count with a sign on it.

the -1 terminal is still worth reaching. wandering costs -1 per step forever, so paying -1 once to
stop is the best thing on offer.

In [25]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=-1,
    terminals={(0, 3): -1},
    walls={(1, 1)},
)
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  -1 
  1   ·   #   ·   · 
  2   ·   ·   ·   · 


In [26]:
policy, V = value_iteration(env, gamma=1.0)

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0 -1.0 -1.0 -1.0  0.0 
   1 -1.0  0.0 -1.0 -1.0 
   2 -1.0 -1.0 -1.0 -1.0 
-------------------------
iter 1: delta=1.000000
  r/c   0    1    2    3 
   0 -2.0 -2.0 -1.0  0.0 
   1 -2.0  0.0 -2.0 -1.0 
   2 -2.0 -2.0 -2.0 -2.0 
-------------------------
iter 2: delta=1.000000
  r/c   0    1    2    3 
   0 -3.0 -2.0 -1.0  0.0 
   1 -3.0  0.0 -2.0 -1.0 
   2 -3.0 -3.0 -3.0 -2.0 
-------------------------
iter 3: delta=1.000000
  r/c   0    1    2    3 
   0 -3.0 -2.0 -1.0  0.0 
   1 -4.0  0.0 -2.0 -1.0 
   2 -4.0 -4.0 -3.0 -2.0 
-------------------------
iter 4: delta=1.000000
  r/c   0    1    2    3 
   0 -3.0 -2.0 -1.0  0.0 
   1 -4.0  0.0 -2.0 -1.0 
   2 -5.0 -4.0 -3.0 -2.0 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 -3.0 -2.0 -1.0  0.0 
   1 -4.0  0.0 -2.0 -1.

### step reward -1, value_iteration V settles at -1/(1-gamma) for cells trapped in walls

cells sealed off from the terminal never reach it, so their value is an endless sum of -1 discounted
by gamma: -1/(1-gamma) = -10 at gamma=0.9.

see what happens if the value at next_cell is already -10: -1 + (0.9 * -10) = -10, so the cell reproduces it. that is the fixed point the sweeps are crawling toward.

it takes 133 sweeps to get there. the value creeps toward -10 geometrically and theta is what
finally stops it — unlike the 3x4 grid, where delta dropped straight to 0.

In [27]:
env = GridWorld(
    rows=5,
    cols=6,
    step_reward=-1,
    terminals={(0, 5): -1},
    walls={(2, 2), (2, 3), (3, 1), (3, 4), (4, 2), (4, 3)},
)
env.render()

 r/c  0   1   2   3   4   5 
  0   ·   ·   ·   ·   ·  -1 
  1   ·   ·   ·   ·   ·   · 
  2   ·   ·   #   #   ·   · 
  3   ·   #   ·   ·   #   · 
  4   ·   ·   #   #   ·   · 


In [28]:
policy, V = value_iteration(env)

V init
  r/c   0    1    2    3    4    5 
   0  0.0  0.0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0  0.0  0.0 
   3  0.0  0.0  0.0  0.0  0.0  0.0 
   4  0.0  0.0  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3    4    5 
   0 -1.0 -1.0 -1.0 -1.0 -1.0  0.0 
   1 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 
   2 -1.0 -1.0  0.0  0.0 -1.0 -1.0 
   3 -1.0  0.0 -1.0 -1.0  0.0 -1.0 
   4 -1.0 -1.0  0.0  0.0 -1.0 -1.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3    4    5 
   0 -1.9 -1.9 -1.9 -1.9 -1.0  0.0 
   1 -1.9 -1.9 -1.9 -1.9 -1.9 -1.0 
   2 -1.9 -1.9  0.0  0.0 -1.9 -1.9 
   3 -1.9  0.0 -1.9 -1.9  0.0 -1.9 
   4 -1.9 -1.9  0.0  0.0 -1.9 -1.9 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3    4    5 
   0 -2.71 -2.71 -2.71 -1.9 -1.0  0.0 
   1 -2.71 -2.71 -2.71 -2.71 -1.9 -1.0 
   2 -2.71 -2.71  0.0  0.0 -2.71 -1.9 
   3 -2.71  0.0 -2.71 -2.71  0.0 -2.71 
   4

### if gamma is 1.0 values bounded by walls keeps updating

at gamma=1.0 that sum has no limit. -1 per step forever does not settle anywhere, so the trapped
cells have no fixed point and delta never falls below theta.

this is the one place the loop needs a guard. max_iters is not a performance knob here, it is the
only thing that ends the run.

In [29]:
def value_iteration(env: GridWorld, gamma=0.9, theta=1e-6, max_iters=1000):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    print('V init')
    show_V(env, V)
    print('-' * 25)  
    policy = [[None for _ in range(env.cols)] for _ in range(env.rows)]
    delta = float("inf")
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                max_value = float("-inf")
                for action in ACTIONS:
                    next_cell, reward = env.step(cell, action)
                    action_value = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                    if action_value > max_value:
                        max_value = action_value
                        policy[r][c] = action
                V[r][c] = max_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        print(f"iter {i}: delta={delta:.6f}")
        show_V(env, V)
        print("-" * 25)
        i += 1
    print('policy:')
    render_policy(env, policy)
    print("-" * 25)
    converged = i < max_iters
    if converged:
        print(f'value_iteration converged in {i} iterations')
    else:
        print(f'value_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [30]:
policy, V, converged = value_iteration(env, gamma=1.0, max_iters=100)

V init
  r/c   0    1    2    3    4    5 
   0  0.0  0.0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0  0.0  0.0 
   3  0.0  0.0  0.0  0.0  0.0  0.0 
   4  0.0  0.0  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3    4    5 
   0 -1.0 -1.0 -1.0 -1.0 -1.0  0.0 
   1 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 
   2 -1.0 -1.0  0.0  0.0 -1.0 -1.0 
   3 -1.0  0.0 -1.0 -1.0  0.0 -1.0 
   4 -1.0 -1.0  0.0  0.0 -1.0 -1.0 
-------------------------
iter 1: delta=1.000000
  r/c   0    1    2    3    4    5 
   0 -2.0 -2.0 -2.0 -2.0 -1.0  0.0 
   1 -2.0 -2.0 -2.0 -2.0 -2.0 -1.0 
   2 -2.0 -2.0  0.0  0.0 -2.0 -2.0 
   3 -2.0  0.0 -2.0 -2.0  0.0 -2.0 
   4 -2.0 -2.0  0.0  0.0 -2.0 -2.0 
-------------------------
iter 2: delta=1.000000
  r/c   0    1    2    3    4    5 
   0 -3.0 -3.0 -3.0 -2.0 -1.0  0.0 
   1 -3.0 -3.0 -3.0 -3.0 -2.0 -1.0 
   2 -3.0 -3.0  0.0  0.0 -3.0 -2.0 
   3 -3.0  0.0 -3.0 -3.0  0.0 -3.0 
   4 -3.0 -3.0  0.

### step reward -1, policy_evaluation hits against the ceiling similar effect as being trapped in walls

cells that keep bumping into the boundary behave like the walled-off ones. the policy sends them
into an edge, they stay put, and the value settles at the same -10.

evaluation reports 133 sweeps on every one of the 7 rounds. the same number each time, because it
restarts from zeros and the cells that can't reach goal under the current policy crawl to -10.

and since cells guarded by walls, have no way to reach the goal, they keep that behaviour until end, while other cells which were hitting the ceiling escape that behaviour as policy updates

In [31]:
policy, V, converged = policy_iteration(env)

iter 0:
policy_evaluation converged in 133 iterations
V:
  r/c   0    1    2    3    4    5 
   0 -10.0 -10.0 -10.0 -10.0 -10.0  0.0 
   1 -10.0 -10.0 -10.0 -10.0 -10.0 -1.0 
   2 -10.0 -10.0  0.0  0.0 -10.0 -1.9 
   3 -10.0  0.0 -10.0 -10.0  0.0 -2.71 
   4 -10.0 -10.0  0.0  0.0 -10.0 -3.44 
-------------------------
actions changed = 4
 r/c  0   1   2   3   4   5 
  0   ↑   ↑   ↑   ↑   →  -1 
  1   ↑   ↑   ↑   ↑   →   ↑ 
  2   ↑   ↑   #   #   →   ↑ 
  3   ↑   #   ↑   ↑   #   ↑ 
  4   ↑   ↑   #   #   →   ↑ 
-----
iter 1:
policy_evaluation converged in 133 iterations
V:
  r/c   0    1    2    3    4    5 
   0 -10.0 -10.0 -10.0 -10.0 -1.0  0.0 
   1 -10.0 -10.0 -10.0 -10.0 -1.9 -1.0 
   2 -10.0 -10.0  0.0  0.0 -2.71 -1.9 
   3 -10.0  0.0 -10.0 -10.0  0.0 -2.71 
   4 -10.0 -10.0  0.0  0.0 -4.1 -3.44 
-------------------------
actions changed = 2
 r/c  0   1   2   3   4   5 
  0   ↑   ↑   ↑   →   →  -1 
  1   ↑   ↑   ↑   →   →   ↑ 
  2   ↑   ↑   #   #   →   ↑ 
  3   ↑   #   ↑   ↑   #   ↑

### evaluation never converged, and policy_iteration found the optimal policy anyway

gamma=1.0 on this grid has no fixed point, so every round hits pe_max_iters=100 with the values
still moving. all 7 of them. early on that is both the sealed-off cells and the ones the policy
walks into a boundary. later only the pocket at (3,2) and (3,3) is left falling — everything that
reaches the goal has settled on its exact value.

policy_iteration converged regardless — same 7 rounds it takes when evaluation does converge, same
policy at the end.

it converges because evaluation still gets the reachable values right. a cell next to the goal
settles on its true value even while the trapped cells keep falling, so improvement hands it the
right action. that action makes the cell before it settle, which lets that one pick its action next
round, and the chain walks outward from the goal — one column per round.

-100 never confuses the argmax either, only the ordering of the values has to be right.

In [32]:
def policy_evaluation(
    env: GridWorld,
    policy: list[list[str | None]],
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=True
):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    if verbose:
        print('V init')
        show_V(env, V)
        print('-' * 25)  
    delta = float("inf")
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                action = policy[r][c]
                next_cell, reward = env.step(cell, action)
                V[r][c] = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            print(f"iter {i}: delta={delta:.6f}")
            show_V(env, V)
            print("-" * 25)
        i += 1
    converged = delta < theta
    if converged:
        print(f'policy_evaluation converged in {i} iterations')
    else:
        print(f"policy_evaluation did not converge in {max_iters} iterations")
    return V, converged

In [33]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    pass_incumbent_policy=True,
    pe_max_iters=1000,
    pe_verbose=False,
):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    policy = [[UP for _ in range(env.cols)] for _ in range(env.rows)]
    changed = None
    i = 0
    while changed != 0 and i < max_iters:
        print(f'iter {i}:')
        V, pe_converged = policy_evaluation(
            env,
            policy,
            gamma,
            theta,
            max_iters=pe_max_iters,
            verbose=pe_verbose,
        )
        new_policy = read_policy(
            env,
            V,
            gamma,
            theta,
            incumbent_policy=policy if pass_incumbent_policy else None,
        )
        changed = 0
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                if new_policy[r][c] != policy[r][c]:
                    changed += 1
        print(f'V:')
        show_V(env, V)
        print('-' * 25)
        print(f'actions changed = {changed}')
        render_policy(env, new_policy)
        print('-' * 5)
        policy = new_policy
        i += 1
    converged = changed == 0
    if converged:
        print(f'policy_iteration converged in {i} iterations')
    else:
        print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [34]:
policy, V, converged = policy_iteration(env, gamma=1.0, pe_max_iters=100)

iter 0:
policy_evaluation did not converge in 100 iterations
V:
  r/c   0    1    2    3    4    5 
   0 -100.0 -100.0 -100.0 -100.0 -100.0  0.0 
   1 -100.0 -100.0 -100.0 -100.0 -100.0 -1.0 
   2 -100.0 -100.0  0.0  0.0 -100.0 -2.0 
   3 -100.0  0.0 -100.0 -100.0  0.0 -3.0 
   4 -100.0 -100.0  0.0  0.0 -100.0 -4.0 
-------------------------
actions changed = 4
 r/c  0   1   2   3   4   5 
  0   ↑   ↑   ↑   ↑   →  -1 
  1   ↑   ↑   ↑   ↑   →   ↑ 
  2   ↑   ↑   #   #   →   ↑ 
  3   ↑   #   ↑   ↑   #   ↑ 
  4   ↑   ↑   #   #   →   ↑ 
-----
iter 1:
policy_evaluation did not converge in 100 iterations
V:
  r/c   0    1    2    3    4    5 
   0 -100.0 -100.0 -100.0 -100.0 -1.0  0.0 
   1 -100.0 -100.0 -100.0 -100.0 -2.0 -1.0 
   2 -100.0 -100.0  0.0  0.0 -3.0 -2.0 
   3 -100.0  0.0 -100.0 -100.0  0.0 -3.0 
   4 -100.0 -100.0  0.0  0.0 -5.0 -4.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3   4   5 
  0   ↑   ↑   ↑   →   →  -1 
  1   ↑   ↑   ↑   →   →   ↑ 
  2   ↑   ↑  

### if both negative step reward and discount is removed, then value_iteration policy degenerates

with step_reward=0 and gamma=1.0 nothing costs anything. every cell that can reach +1 is worth
exactly 1.0, whether it is one step away or ten.

V is still correct. the policy read off it is not: at (2,3) the greedy action is RIGHT, which walks
into the edge and stays there forever. reaching the goal scores no better than never reaching it,
so the argmax is free to pick a self-loop.

urgency is what a discount or a step cost buys. without either, optimal path can't be discovered.

In [35]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


In [36]:
policy, V, converged = value_iteration(env, gamma=1.0)

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  1.0  1.0  0.0 
   1  0.0  0.0  1.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=1.000000
  r/c   0    1    2    3 
   0  1.0  1.0  1.0  0.0 
   1  0.0  0.0  1.0  0.0 
   2  0.0  0.0  1.0  0.0 
-------------------------
iter 3: delta=1.000000
  r/c   0    1    2    3 
   0  1.0  1.0  1.0  0.0 
   1  1.0  0.0  1.0  0.0 
   2  0.0  1.0  1.0  1.0 
-------------------------
iter 4: delta=1.000000
  r/c   0    1    2    3 
   0  1.0  1.0  1.0  0.0 
   1  1.0  0.0  1.0  0.0 
   2  1.0  1.0  1.0  1.0 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0  1.0  1.0  1.0  0.0 
   1  1.0  0.0  1.0  0.

### pass previous V to policy_evaluation

policy_evaluation restarts from zeros on every call, so each round re-derives values close to the
ones it just threw away. warm starting from the previous V should cut that.

on the 3x4 grid: 2, 5, 5, 6 = 18 sweeps cold, against 2, 5, 2, 3 = 12 warm. same 4 rounds, same
policy.

In [37]:
def policy_evaluation(
    env: GridWorld,
    policy: list[list[str | None]],
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    V: list[list[float]] | None = None,
    verbose=True
):
    if V is None:
        V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    if verbose:
        print('V init')
        show_V(env, V)
        print('-' * 25)  
    delta = float("inf")
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                action = policy[r][c]
                next_cell, reward = env.step(cell, action)
                V[r][c] = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            print(f"iter {i}: delta={delta:.6f}")
            show_V(env, V)
            print("-" * 25)
        i += 1
    converged = delta < theta
    if converged:
        print(f'policy_evaluation converged in {i} iterations')
    else:
        print(f"policy_evaluation did not converge in {max_iters} iterations")
    return V, converged, i

In [38]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    pass_incumbent_policy=True,
    pe_max_iters=1000,
    pe_warm_start=True,
    pe_verbose=False,
):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    policy = [[UP for _ in range(env.cols)] for _ in range(env.rows)]
    pe_total_iters = 0
    changed = None
    i = 0
    while changed != 0 and i < max_iters:
        print(f'iter {i}:')
        V, pe_converged, pe_iters = policy_evaluation(
            env,
            policy,
            gamma,
            theta,
            max_iters=pe_max_iters,
            V=V if pe_warm_start else None,
            verbose=pe_verbose,
        )
        pe_total_iters += pe_iters
        new_policy = read_policy(
            env,
            V,
            gamma,
            theta,
            incumbent_policy=policy if pass_incumbent_policy else None,
        )
        changed = 0
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                if new_policy[r][c] != policy[r][c]:
                    changed += 1
        print(f'V:')
        show_V(env, V)
        print('-' * 25)
        print(f'actions changed = {changed}')
        render_policy(env, new_policy)
        print('-' * 5)
        policy = new_policy
        i += 1
    print(f'policy_evaluation - total iterations: {pe_total_iters}')
    converged = changed == 0
    if converged:
        print(f'policy_iteration converged in {i} iterations')
    else:
        print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [39]:
policy, V, converged = policy_iteration(env, pe_warm_start=False)

iter 0:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-----
iter 1:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-----
iter 2:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-----
iter 3:
policy_evaluation converged in 6 iterations
V:
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.7

In [40]:
policy, V, converged = policy_iteration(env, pe_warm_start=True)

iter 0:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-----
iter 1:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-----
iter 2:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-----
iter 3:
policy_evaluation converged in 3 iterations
V:
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.7

### warm start on the walled grid, where the cold cost is real

18 against 12 is a small saving because evaluation was cheap to begin with. the 5x6 grid is where
it shows.

cold: 133 sweeps, seven times over = 931.
warm: 133 once, then 2, 2, 2, 3, 4, 2 = 148.

only the first round pays the crawl to -10. after that the ceiling is already sitting in V, and each
round only repairs the few cells the new policy moved.

In [41]:
env = GridWorld(
    rows=5,
    cols=6,
    step_reward=-1,
    terminals={(0, 5): -1},
    walls={(2, 2), (2, 3), (3, 1), (3, 4), (4, 2), (4, 3)},
)
env.render()

 r/c  0   1   2   3   4   5 
  0   ·   ·   ·   ·   ·  -1 
  1   ·   ·   ·   ·   ·   · 
  2   ·   ·   #   #   ·   · 
  3   ·   #   ·   ·   #   · 
  4   ·   ·   #   #   ·   · 


In [42]:
policy, V, converged = policy_iteration(env, pe_warm_start=False)

iter 0:
policy_evaluation converged in 133 iterations
V:
  r/c   0    1    2    3    4    5 
   0 -10.0 -10.0 -10.0 -10.0 -10.0  0.0 
   1 -10.0 -10.0 -10.0 -10.0 -10.0 -1.0 
   2 -10.0 -10.0  0.0  0.0 -10.0 -1.9 
   3 -10.0  0.0 -10.0 -10.0  0.0 -2.71 
   4 -10.0 -10.0  0.0  0.0 -10.0 -3.44 
-------------------------
actions changed = 4
 r/c  0   1   2   3   4   5 
  0   ↑   ↑   ↑   ↑   →  -1 
  1   ↑   ↑   ↑   ↑   →   ↑ 
  2   ↑   ↑   #   #   →   ↑ 
  3   ↑   #   ↑   ↑   #   ↑ 
  4   ↑   ↑   #   #   →   ↑ 
-----
iter 1:
policy_evaluation converged in 133 iterations
V:
  r/c   0    1    2    3    4    5 
   0 -10.0 -10.0 -10.0 -10.0 -1.0  0.0 
   1 -10.0 -10.0 -10.0 -10.0 -1.9 -1.0 
   2 -10.0 -10.0  0.0  0.0 -2.71 -1.9 
   3 -10.0  0.0 -10.0 -10.0  0.0 -2.71 
   4 -10.0 -10.0  0.0  0.0 -4.1 -3.44 
-------------------------
actions changed = 2
 r/c  0   1   2   3   4   5 
  0   ↑   ↑   ↑   →   →  -1 
  1   ↑   ↑   ↑   →   →   ↑ 
  2   ↑   ↑   #   #   →   ↑ 
  3   ↑   #   ↑   ↑   #   ↑

In [43]:
policy, V, converged = policy_iteration(env, pe_warm_start=True)

iter 0:
policy_evaluation converged in 133 iterations
V:
  r/c   0    1    2    3    4    5 
   0 -10.0 -10.0 -10.0 -10.0 -10.0  0.0 
   1 -10.0 -10.0 -10.0 -10.0 -10.0 -1.0 
   2 -10.0 -10.0  0.0  0.0 -10.0 -1.9 
   3 -10.0  0.0 -10.0 -10.0  0.0 -2.71 
   4 -10.0 -10.0  0.0  0.0 -10.0 -3.44 
-------------------------
actions changed = 4
 r/c  0   1   2   3   4   5 
  0   ↑   ↑   ↑   ↑   →  -1 
  1   ↑   ↑   ↑   ↑   →   ↑ 
  2   ↑   ↑   #   #   →   ↑ 
  3   ↑   #   ↑   ↑   #   ↑ 
  4   ↑   ↑   #   #   →   ↑ 
-----
iter 1:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3    4    5 
   0 -10.0 -10.0 -10.0 -10.0 -1.0  0.0 
   1 -10.0 -10.0 -10.0 -10.0 -1.9 -1.0 
   2 -10.0 -10.0  0.0  0.0 -2.71 -1.9 
   3 -10.0  0.0 -10.0 -10.0  0.0 -2.71 
   4 -10.0 -10.0  0.0  0.0 -4.1 -3.44 
-------------------------
actions changed = 2
 r/c  0   1   2   3   4   5 
  0   ↑   ↑   ↑   →   →  -1 
  1   ↑   ↑   ↑   →   →   ↑ 
  2   ↑   ↑   #   #   →   ↑ 
  3   ↑   #   ↑   ↑   #   ↑ 
